# Notebook 3 — LSTM Autoencoder

Trains the LSTM autoencoder for sequential/drift anomaly detection (Paper F1=0.90).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.data_generator import generate_dataset
from src.preprocessor   import PipelinePreprocessor
from src.models         import LSTMAutoencoderDetector

In [ ]:
df   = generate_dataset(seed=42)
prep = PipelinePreprocessor(window_size=10)
X    = prep.fit_transform(df)
X_seq = prep.make_sequences(X)
y    = df['is_anomaly'].values
# Sequences start at index 9 (window_size-1)
y_seq = y[9:]
print(f'X_seq shape: {X_seq.shape}, anomalies in seqs: {y_seq.sum()}')

In [ ]:
lstm = LSTMAutoencoderDetector(window_size=10)
lstm.fit(X_seq[y_seq == 0], epochs=20)  # Use 20 epochs for notebook speed

In [ ]:
result = lstm.evaluate(X_seq, y_seq)
print(f"LSTM Precision: {result['precision']:.3f}  Recall: {result['recall']:.3f}  F1: {result['f1']:.3f}")

In [ ]:
scores = lstm.score_samples(X_seq)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(scores, color='steelblue', linewidth=0.8)
axes[0].axhline(lstm.threshold_, color='red', linestyle='--', label=f'Threshold = {lstm.threshold_:.4f}')
axes[0].scatter(np.where(y_seq==1)[0], scores[y_seq==1], c='red', s=50, zorder=5, label='True Anomaly')
axes[0].set(ylabel='Reconstruction Error (MAE)', title='LSTM Autoencoder — Reconstruction Error')
axes[0].legend()

# Show training history if available
if hasattr(lstm, '_history') and lstm._history is not None:
    axes[1].plot(lstm._history.history['loss'], label='Train Loss')
    if 'val_loss' in lstm._history.history:
        axes[1].plot(lstm._history.history['val_loss'], label='Val Loss')
    axes[1].set(xlabel='Epoch', ylabel='MAE Loss', title='LSTM Training History')
    axes[1].legend()

plt.tight_layout()
plt.savefig('../results/nb_lstm_model.png', dpi=150)
plt.show()